# U-Net baseline — resultado parcial (estágio 10)

Treina uma U-Net compacta sobre o conjunto experimental disponível em `MyDrive/tcc/imagens`, seleciona o melhor checkpoint pela IoU de validação e gera métricas e figuras de teste. O objetivo desta etapa é validar o pipeline completo com um resultado parcial real antes da integração definitiva da U-Mamba.

## Bootstrap

Obtém a versão atual do código compartilhado diretamente do repositório oficial do TCC.

In [ ]:
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/oguel/tcc-umamba/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)
workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")


## Imports, seed e dispositivo

Carrega as rotinas reutilizáveis, fixa a seed e seleciona GPU quando disponível.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader

from src import io
from src.config import get_config
from src.data.dataset import CoffeeSegmentationDataset
from src.losses import BCEDiceLoss
from src.models.unet import UNet
from src.trainer import fit_model, run_epoch
from src.utils import set_all_seeds

config = get_config()
set_all_seeds(int(config["reproducibility"]["seed"]))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## Dataset e DataLoaders

Carrega os subconjuntos já separados pelo professor. O conjunto de treino recebe augmentations leves; validação e teste permanecem determinísticos.

In [ ]:
storage_paths = io.resolve_storage_paths()
dataset_root = storage_paths["data_embrapa"]
image_size = int(config["data"]["patch_size"])

train_dataset = CoffeeSegmentationDataset(
    dataset_root / "Imagens_treino",
    dataset_root / "Mascaras_treino",
    image_size=image_size,
    augment=True,
)
val_dataset = CoffeeSegmentationDataset(
    dataset_root / "Imagens_validacao",
    dataset_root / "Mascaras_validacao",
    image_size=image_size,
)
test_dataset = CoffeeSegmentationDataset(
    dataset_root / "Imagens_teste",
    dataset_root / "Mascaras_teste",
    image_size=image_size,
)

batch_size = int(config["training"]["batch_size"])
pin_memory = device.type == "cuda"
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_memory)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_memory)

print(f"Treino: {len(train_dataset)} | Validação: {len(val_dataset)} | Teste: {len(test_dataset)}")


## Modelo, loss e otimizador

Inicializa a U-Net baseline com entrada RGB, BCE + Dice Loss e AdamW.

In [ ]:
channels = tuple(int(value) for value in config["model"]["unet_channels"])
model = UNet(
    in_channels=int(config["model"]["input_channels_experimental"]),
    out_channels=int(config["model"]["output_channels"]),
    channels=channels,
).to(device)

criterion = BCEDiceLoss(
    bce_weight=float(config["loss"]["bce_weight"]),
    dice_weight=float(config["loss"]["dice_weight"]),
)
optimizer = AdamW(model.parameters(), lr=float(config["training"]["lr"]), weight_decay=1e-4)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Parâmetros treináveis: {parameter_count:,}")


## Treinamento parcial

Executa um número reduzido de épocas, suficiente para um resultado preliminar. O melhor checkpoint é escolhido pela IoU de validação.

In [ ]:
epochs = int(config["training"].get("partial_epochs", 10))
checkpoint_path = storage_paths["models_unet"] / "best_unet_experimental.pt"

history = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    checkpoint_path=checkpoint_path,
)


## Curvas de treinamento

Salva loss, IoU, F1, tempo por época e pico de VRAM no Google Drive.

In [ ]:
history_df = pd.DataFrame(history)
metrics_dir = storage_paths["artifacts_metrics"] / "unet"
figures_dir = storage_paths["artifacts_figures"] / "unet"
metrics_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

history_path = metrics_dir / "experimental_history.csv"
history_df.to_csv(history_path, index=False)

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="treino")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="validação")
axes[0].set_title("Loss")
axes[0].set_xlabel("Época")
axes[0].legend()
axes[1].plot(history_df["epoch"], history_df["val_iou"], label="IoU")
axes[1].plot(history_df["epoch"], history_df["val_f1"], label="F1")
axes[1].set_title("Validação")
axes[1].set_xlabel("Época")
axes[1].legend()
figure.tight_layout()
curves_path = figures_dir / "experimental_training_curves.png"
figure.savefig(curves_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Histórico: {history_path}")
print(f"Curvas: {curves_path}")


## Avaliação no teste

Restaura o melhor checkpoint e calcula as métricas agregadas no conjunto de teste.

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
test_metrics = run_epoch(model, test_loader, criterion, device)

test_metrics.update({
    "best_epoch": int(checkpoint["epoch"]),
    "best_val_iou": float(checkpoint["val_iou"]),
    "parameters": int(parameter_count),
})

metrics_path = metrics_dir / "experimental_test_metrics.json"
metrics_path.write_text(json.dumps(test_metrics, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(test_metrics, indent=2, ensure_ascii=False))
print(f"Métricas salvas em: {metrics_path}")


## Resultado visual

Gera a figura principal do resultado parcial: imagem original, ground truth e previsão binária.

In [ ]:
model.eval()
sample_count = min(3, len(test_dataset))
figure, axes = plt.subplots(sample_count, 3, figsize=(12, 4 * sample_count))
if sample_count == 1:
    axes = axes.reshape(1, -1)

with torch.inference_mode():
    for row in range(sample_count):
        image, mask = test_dataset[row]
        logits = model(image.unsqueeze(0).to(device))
        prediction = (torch.sigmoid(logits)[0, 0] >= 0.5).cpu().numpy()

        axes[row, 0].imshow(image.permute(1, 2, 0).numpy())
        axes[row, 0].set_title("Imagem")
        axes[row, 1].imshow(mask[0].numpy(), cmap="gray", vmin=0, vmax=1)
        axes[row, 1].set_title("Ground Truth")
        axes[row, 2].imshow(prediction, cmap="gray", vmin=0, vmax=1)
        axes[row, 2].set_title("Predição U-Net")
        for col in range(3):
            axes[row, col].axis("off")

figure.tight_layout()
prediction_path = figures_dir / "experimental_test_predictions.png"
figure.savefig(prediction_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Resultado visual salvo em: {prediction_path}")


## Interpretação correta

Este experimento é preliminar. As métricas servem para provar que a cadeia de dados, treinamento, inferência e avaliação está funcional. Elas não devem ser tratadas como desempenho definitivo da U-Mamba nem como conclusão final do TCC.